<a href="https://colab.research.google.com/github/komal-shahzada/ArchTechnologies/blob/main/minst_handwritten_digit_recogination_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importing libraries for Deep Learning and Data Visualization
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import gradio as gr

print("TensorFlow and Gradio loaded successfully!")

TensorFlow and Gradio loaded successfully!


In [2]:
# Loading the MNIST dataset
mnist = tf.keras.datasets.mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalizing pixel values to be between 0 and 1 for faster training
X_train, X_test = X_train / 255.0, X_test / 255.0

# Reshaping data to include the channel dimension (required for CNNs)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

print(f"Training data shape: {X_train.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training data shape: (60000, 28, 28, 1)


In [3]:
# Defining the CNN architecture
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax') # 10 outputs for digits 0-9
])

# Compiling the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("CNN Model architecture is ready.")

CNN Model architecture is ready.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
# Training the model
history = model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

# Evaluating the final accuracy
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nFinal Test Accuracy: {test_acc*100:.2f}%")

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 60s 31ms/step - accuracy: 0.9590 - loss: 0.1368 - val_accuracy: 0.9836 - val_loss: 0.0579
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 54s 29ms/step - accuracy: 0.9859 - loss: 0.0448 - val_accuracy: 0.9878 - val_loss: 0.0426
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 54s 29ms/step - accuracy: 0.9898 - loss: 0.0313 - val_accuracy: 0.9890 - val_loss: 0.0331
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 81s 29ms/step - accuracy: 0.9927 - loss: 0.0231 - val_accuracy: 0.9904 - val_loss: 0.0289
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 56s 30ms/step - accuracy: 0.9945 - loss: 0.0170 - val_accuracy: 0.9902 - val_loss: 0.0309
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9902 - loss: 0.0309

Final Test Accuracy: 99.02%


In [6]:
from PIL import Image

def classify_digit(img):
    # Gradio Sketchpad can sometimes provide a dictionary with 'composite' key
    # or just a numpy array. We handle both cases here.
    if isinstance(img, dict):
        img = img['composite']

    # Convert to PIL Image to easily resize and convert to grayscale
    img = Image.fromarray(img.astype('uint8')).convert('L')
    img = img.resize((28, 28))

    # Convert back to numpy array and normalize
    img_array = np.array(img).reshape(1, 28, 28, 1) / 255.0

    # Since background in Sketchpad is often white and digits are black,
    # and MNIST is vice-versa, we might need to invert colors if accuracy is low.
    # img_array = 1.0 - img_array

    prediction = model.predict(img_array).flatten()

    # Returning the probability for each digit
    return {str(i): float(prediction[i]) for i in range(10)}

# Relaunch the interface
label = gr.Label(num_top_classes=3)
interface = gr.Interface(
    fn=classify_digit,
    inputs=gr.Sketchpad(label="Draw a Digit (0-9)"),
    outputs=label,
    title="Handwritten Digit Recognizer - Task 2"
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8ddd0b5e24794c01bc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
